# Stress Prediction — Leave-One-Out 피처 Ablation

각 피처를 하나씩 빼면서 5-Fold CV MAE가 어떻게 바뀌는지 자동으로 측정합니다.

- **빼면 MAE가 올라감(나빠짐)** -> 그 피처는 유용함 -> 유지
- **빼면 MAE가 내려감(좋아짐)** -> 그 피처는 노이즈/방해 -> 제거 후보
- **거의 안 바뀜** -> 있으나 없으나 무관

`mean_working`을 포함한 모든 원본/파생 컬럼을 대상으로 합니다. 속도를 위해 트리 개수는
줄였고(탐색용), 최종 후보가 정해지면 원래 `n_estimators=1200`으로 확인하는 걸 권장합니다.


In [1]:

from __future__ import annotations

import os
from itertools import combinations
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold

TARGET = "stress_score"
ID_COLUMN = "ID"
RANDOM_SEED = 42
N_ESTIMATORS_SEARCH = 500  # 탐색용으로 축소 (원본 1200)
TREE_QUANTILE = 0.54
PAIR_QUANTILE = 0.48
PAIR_WEIGHT = 0.28

DATA_DIR = Path("/Users/bitsaem/Desktop/stress_project/open")

FEATURE_COPY_COUNTS = {
    "mean_working": 1, "bmi": 4, "cholesterol": 2, "height": 2,
    "glucose": 3, "weight": 2, "cholesterol_glucose_ratio": 2, "bone_density": 4,
}

PAIR_COLUMNS = [
    "mean_working", "bmi", "cholesterol", "height",
    "glucose", "weight", "cholesterol_glucose_ratio", "bone_density",
]


def add_features(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    height_m = result["height"] / 100.0
    sbp = result["systolic_blood_pressure"]
    dbp = result["diastolic_blood_pressure"]
    result["bmi"] = result["weight"] / height_m.pow(2)
    result["pulse_pressure"] = sbp - dbp
    result["mean_arterial_pressure"] = (sbp + 2.0 * dbp) / 3.0
    result["blood_pressure_ratio"] = sbp / (dbp + 1e-6)
    result["cholesterol_glucose_ratio"] = result["cholesterol"] / (result["glucose"] + 1e-6)
    result["missing_count"] = result.isna().sum(axis=1)
    return result


def add_feature_copies(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    for feature, n_copies in FEATURE_COPY_COUNTS.items():
        if feature not in result.columns:
            continue  # 해당 피처가 이번 ablation에서 제거된 경우 복사도 건너뜀
        for i in range(1, n_copies + 1):
            result[f"{feature}__copy{i}"] = result[feature]
    return result


def make_preprocessor(frame: pd.DataFrame) -> ColumnTransformer:
    categorical = frame.select_dtypes(include=["object", "category", "string"]).columns.tolist()
    numerical = frame.select_dtypes(include=["number"]).columns.tolist()
    return ColumnTransformer([
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), categorical),
        ("numerical", SimpleImputer(strategy="median", add_indicator=True), numerical),
    ])


def tree_prediction(train_features, val_features, target) -> np.ndarray:
    weighted_train = add_feature_copies(train_features)
    weighted_val = add_feature_copies(val_features)
    preprocessor = make_preprocessor(weighted_train)
    train_matrix = preprocessor.fit_transform(weighted_train)
    val_matrix = preprocessor.transform(weighted_val)
    model = ExtraTreesRegressor(
        n_estimators=N_ESTIMATORS_SEARCH, min_samples_leaf=1, max_features=1,
        criterion="squared_error", random_state=RANDOM_SEED, n_jobs=-1,
    )
    model.fit(train_matrix, target)
    tree_predictions = np.column_stack([t.predict(val_matrix) for t in model.estimators_])
    return np.quantile(tree_predictions, TREE_QUANTILE, axis=1)


def empirical_rank_transform(train_values, test_values):
    median = np.nanmedian(train_values)
    train_values = np.nan_to_num(train_values, nan=median)
    test_values = np.nan_to_num(test_values, nan=median)
    sorted_train = np.sort(train_values)
    train_rank = np.searchsorted(sorted_train, train_values, side="right") / len(train_values)
    test_rank = np.searchsorted(sorted_train, test_values, side="right") / len(train_values)
    return train_rank.astype(np.float32), test_rank.astype(np.float32)


def pair_prediction(train_features, val_features, target, pair_columns) -> np.ndarray:
    train_rank, val_rank = {}, {}
    for column in pair_columns:
        train_rank[column], val_rank[column] = empirical_rank_transform(
            train_features[column].to_numpy(dtype=float),
            val_features[column].to_numpy(dtype=float),
        )
    target_values = target.to_numpy(dtype=float)
    neighbor_targets = []
    for first, second in combinations(pair_columns, 2):
        distance = (
            np.abs(val_rank[first][:, None] - train_rank[first][None, :])
            + np.abs(val_rank[second][:, None] - train_rank[second][None, :])
        )
        nearest_idx = np.argmin(distance, axis=1)
        neighbor_targets.append(target_values[nearest_idx])
    return np.quantile(np.column_stack(neighbor_targets), PAIR_QUANTILE, axis=1)


## 1. 데이터 로드 + 기준(baseline) CV MAE

In [2]:

train = pd.read_csv(DATA_DIR / "train.csv")
raw_X = add_features(train.drop(columns=[TARGET, ID_COLUMN])).reset_index(drop=True)
y = train[TARGET].reset_index(drop=True)

all_columns = raw_X.columns.tolist()
cv = KFold(5, shuffle=True, random_state=RANDOM_SEED)


def cv_mae_for_columns(tree_drop_cols, pair_columns):
    """tree_drop_cols: tree 모델 입력에서 뺄 컬럼 목록. pair_columns: pair 방식에 쓸 컬럼 목록."""
    oof = np.zeros(len(y))
    for train_idx, val_idx in cv.split(raw_X):
        train_feat = raw_X.iloc[train_idx].reset_index(drop=True).drop(columns=tree_drop_cols, errors="ignore")
        val_feat = raw_X.iloc[val_idx].reset_index(drop=True).drop(columns=tree_drop_cols, errors="ignore")
        y_train = y.iloc[train_idx].reset_index(drop=True)

        pred_tree = tree_prediction(train_feat, val_feat, y_train)
        pred_pair = pair_prediction(
            raw_X.iloc[train_idx].reset_index(drop=True),
            raw_X.iloc[val_idx].reset_index(drop=True),
            y_train, pair_columns,
        )
        combined = np.clip(np.round((1 - PAIR_WEIGHT) * pred_tree + PAIR_WEIGHT * pred_pair, 2), 0.0, 1.0)
        oof[val_idx] = combined
    return mean_absolute_error(y, oof)


baseline_mae = cv_mae_for_columns(tree_drop_cols=[], pair_columns=PAIR_COLUMNS)
print(f"### Baseline CV MAE (탐색용 n_estimators={N_ESTIMATORS_SEARCH}): {baseline_mae:.6f}")


### Baseline CV MAE (탐색용 n_estimators=500): 0.148297


## 2. Leave-One-Out — tree 모델에서 컬럼 하나씩 제거

각 컬럼을 tree 모델 입력에서만 빼봅니다 (pair 방식은 `PAIR_COLUMNS` 그대로 유지).
`delta`가 **음수**면 그 컬럼을 빼는 게 더 좋다는 뜻입니다 (제거 후보).


In [3]:

loo_rows = []
for col in all_columns:
    mae = cv_mae_for_columns(tree_drop_cols=[col], pair_columns=PAIR_COLUMNS)
    loo_rows.append({"removed_column": col, "cv_mae": mae, "delta_vs_baseline": mae - baseline_mae})
    print(f"{col}: {mae:.6f} (delta {mae - baseline_mae:+.6f})")

loo_df = pd.DataFrame(loo_rows).sort_values("delta_vs_baseline")
print()
print("### 빼면 좋아지는(delta 음수) 컬럼 -> 제거 후보:")
display(loo_df[loo_df.delta_vs_baseline < 0])
print()
print("### 전체 결과 (delta 오름차순):")
display(loo_df)


gender: 0.147790 (delta -0.000507)
age: 0.148310 (delta +0.000013)
height: 0.148103 (delta -0.000193)
weight: 0.148143 (delta -0.000153)
cholesterol: 0.148287 (delta -0.000010)
systolic_blood_pressure: 0.148233 (delta -0.000063)
diastolic_blood_pressure: 0.148650 (delta +0.000353)
glucose: 0.148357 (delta +0.000060)
bone_density: 0.148553 (delta +0.000257)
activity: 0.147980 (delta -0.000317)
smoke_status: 0.148133 (delta -0.000163)
medical_history: 0.147820 (delta -0.000477)
family_medical_history: 0.148647 (delta +0.000350)
sleep_pattern: 0.148043 (delta -0.000253)
edu_level: 0.149290 (delta +0.000993)
mean_working: 0.148313 (delta +0.000017)
bmi: 0.147917 (delta -0.000380)
pulse_pressure: 0.148493 (delta +0.000197)
mean_arterial_pressure: 0.148147 (delta -0.000150)
blood_pressure_ratio: 0.148480 (delta +0.000183)
cholesterol_glucose_ratio: 0.147953 (delta -0.000343)
missing_count: 0.148013 (delta -0.000283)

### 빼면 좋아지는(delta 음수) 컬럼 -> 제거 후보:


,removed_column,cv_mae,delta_vs_baseline
0,gender,0.147790,-0.000507
11,medical_history,0.147820,-0.000477
16,bmi,0.147917,-0.000380
20,cholesterol_glucose_ratio,0.147953,-0.000343
9,activity,0.147980,-0.000317
21,missing_count,0.148013,-0.000283
13,sleep_pattern,0.148043,-0.000253
2,height,0.148103,-0.000193
10,smoke_status,0.148133,-0.000163
3,weight,0.148143,-0.000153



### 전체 결과 (delta 오름차순):


,removed_column,cv_mae,delta_vs_baseline
0,gender,0.147790,-0.000507
11,medical_history,0.147820,-0.000477
16,bmi,0.147917,-0.000380
20,cholesterol_glucose_ratio,0.147953,-0.000343
9,activity,0.147980,-0.000317
21,missing_count,0.148013,-0.000283
13,sleep_pattern,0.148043,-0.000253
2,height,0.148103,-0.000193
10,smoke_status,0.148133,-0.000163
3,weight,0.148143,-0.000153


## 3. mean_working 포함/제외 x pair_columns 조합 테스트

1등 팀이 언급한 방식대로, `mean_working`을 살리거나 죽인 상태에서 pair_columns 조합을 바꿔가며 비교합니다.


In [4]:

candidate_extra_cols = [c for c in PAIR_COLUMNS if c != "mean_working"]

combo_rows = []
for keep_mean_working in [True, False]:
    base_pair_cols = ["mean_working"] + candidate_extra_cols if keep_mean_working else candidate_extra_cols
    mae_full = cv_mae_for_columns(tree_drop_cols=[], pair_columns=base_pair_cols)
    combo_rows.append({
        "mean_working": "포함" if keep_mean_working else "제외",
        "pair_columns": "전체", "cv_mae": mae_full,
    })
    for col in candidate_extra_cols:
        cols = (["mean_working"] if keep_mean_working else []) + [col]
        if len(cols) < 2:
            continue
        mae = cv_mae_for_columns(tree_drop_cols=[], pair_columns=cols)
        combo_rows.append({
            "mean_working": "포함" if keep_mean_working else "제외",
            "pair_columns": col, "cv_mae": mae,
        })
    print(f"mean_working={'포함' if keep_mean_working else '제외'} 완료")

combo_df = pd.DataFrame(combo_rows).sort_values("cv_mae")
print()
print(f"(참고) v34 원본 PAIR_COLUMNS 전체 사용 CV MAE(탐색용 트리개수 기준): {baseline_mae:.6f}")
display(combo_df)


mean_working=포함 완료
mean_working=제외 완료

(참고) v34 원본 PAIR_COLUMNS 전체 사용 CV MAE(탐색용 트리개수 기준): 0.148297


,mean_working,pair_columns,cv_mae
0,포함,전체,0.148297
8,제외,전체,0.149460
2,포함,cholesterol,0.175523
5,포함,weight,0.175873
4,포함,glucose,0.177023
3,포함,height,0.179430
1,포함,bmi,0.179700
6,포함,cholesterol_glucose_ratio,0.179733
7,포함,bone_density,0.184720


## 다음 단계

1번(Baseline), 2번(LOO), 3번(mean_working 조합) 결과에서 가장 낮은 `cv_mae`를 주는 설정을 찾아
`FINAL` 노트북의 `PAIR_COLUMNS`와 `add_features`/tree 입력 컬럼을 그 조합으로 바꾼 뒤,
`N_ESTIMATORS`를 1200으로 되돌려서 최종 제출 파일을 다시 생성하세요.
